# H5: CNN + Transformer: помогает ли self-attention?

**Проверяемая гипотеза:** Прирост exact match к CRNN base минимум 2 п.п.

Сходный свёрточный extractor → positional encoding → Transformer Encoder → linear → CTC. Attention связывает все позиции последовательности; явной рекуррентной памяти LSTM нет. Готовые CRNN и Transformer обучались отдельно: результат не доказывает чистый причинный эффект attention.

[Теория и исследования](../research/THEORY_AND_PAPERS.md). [Зафиксированный протокол](../research/PROTOCOL.md). Пороговые значения — наши инженерные критерии, не требования заказчика.

In [1]:
from pathlib import Path
import sys, json
ROOT=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/'captcha_lab').is_dir())
if str(ROOT) not in sys.path:sys.path.insert(0,str(ROOT))
import pandas as pd
from captcha_lab.evaluation import (manifest,run_local,metrics,scored,
                                      compare,all_summaries,hypothesis_results,RESULTS)
audit=json.loads((ROOT/'datasets/captcha-images/audit.json').read_text())
assert audit['holdout_images']==1984 and audit['shared_images']==200
display(pd.DataFrame([{'split':k,'n':v} for k,v in audit['source_split_counts'].items()]))
print('Holdout manifest SHA-256:',audit['holdout_manifest_sha256'])
print('Shared manifest SHA-256:',audit['shared_manifest_sha256'])
print('No prediction caches: inference cell below executes the actual model.')

,split,n
0,train,6000
1,validation,2000
2,test,2000


Holdout manifest SHA-256: 33fe043e8dec7b0e238c6bc81bb63c601699540b4b6c2025ad9480e21164f42b
Shared manifest SHA-256: 675bc9b4d93369e68817412fa08a72c6e21de23c820e00e42bc30d06c7390384
No prediction caches: inference cell below executes the actual model.


## Размеченная выборка

Публичный полный test содержит 2000 изображений; 16 просмотренных в предыдущем пилоте исключены. Метки и архивы скачаны полностью, не из preview. Функция распознавания получает только пиксели; метки присоединяются после предсказания. Исходные семь целевых меток выделены отдельно, два спорных примера исключены.

In [2]:
labels=manifest('shared_200' if 'conv-transformer-base'=='gemini' else 'holdout')
display(labels[['image_id','label','image_path']].head(12))
print('Evaluation images:',len(labels))

,image_id,label,image_path
0,test_000c1b6d4e5f11fb75d55a7c094b901e,161660,data/benchmark/images/test/161660.000c1b6d4e5f...
1,test_00105a4c053053509955d95368312987,496670,data/benchmark/images/test/496670.00105a4c0530...
2,test_001a3b40ef793b5984df8969f145d7bb,744782,data/benchmark/images/test/744782.001a3b40ef79...
3,test_0021e642491cf5bbc13cfc59fa46084a,017232,data/benchmark/images/test/017232.0021e642491c...
4,test_0047646a0e3b67ebc360e43ba88d1596,853091,data/benchmark/images/test/853091.0047646a0e3b...
5,test_0064b2188592a16991d5394b3ec5bf01,959686,data/benchmark/images/test/959686.0064b2188592...
6,test_00a5d81ab5060a0d45608169ce20bda9,159645,data/benchmark/images/test/159645.00a5d81ab506...
7,test_00e02c7c9da2cc0d3cdf5e3d9183a9f5,952376,data/benchmark/images/test/952376.00e02c7c9da2...
8,test_0123f4327a176c460aeefecd1d69bc11,374639,data/benchmark/images/test/374639.0123f4327a17...
9,test_0127972a796c1f306fdb5f919a7649fa,517493,data/benchmark/images/test/517493.0127972a796c...


Evaluation images: 1984


## Реальный прогон

Ячейка заново загружает модель и выполняет inference на каждой картинке. Ошибки сохраняются, готовые ответы не подставляются. CPU, batch=1, отдельный прогрев на validation.

In [3]:
frames=run_local('conv-transformer-base')

C:\Users\cvind\Documents\Codex\2026-10-21\files-mentioned-by-the-user-c\work\Captcha-MAI\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


conv-transformer-base holdout 200 / 1984


conv-transformer-base holdout 400 / 1984


conv-transformer-base holdout 600 / 1984


conv-transformer-base holdout 800 / 1984


conv-transformer-base holdout 1000 / 1984


conv-transformer-base holdout 1200 / 1984


conv-transformer-base holdout 1400 / 1984


conv-transformer-base holdout 1600 / 1984


conv-transformer-base holdout 1800 / 1984


In [4]:
rows=[]
for dataset,predictions in frames.items():
    labels=manifest(dataset)
    rows.append(dict(dataset=dataset,**metrics(predictions,labels)))
    joined=scored(predictions,labels)
    print(dataset,'— ошибки по изображениям:')
    display(joined.loc[~joined.correct,['image_id','label','prediction','error','error_detail']].head(20))
display(pd.DataFrame(rows))

holdout — ошибки по изображениям:


,image_id,label,prediction,error,error_detail
4,test_0047646a0e3b67ebc360e43ba88d1596,853091,853O9,,
5,test_0064b2188592a16991d5394b3ec5bf01,959686,59686,,
7,test_00e02c7c9da2cc0d3cdf5e3d9183a9f5,952376,95236,,
9,test_0127972a796c1f306fdb5f919a7649fa,517493,5I7493,,
11,test_0159b035fbb7d58b185c92b964eec949,130021,130027,,
15,test_0214693960836c1e7500ee8ed61de40a,906192,OOOO2,,
16,test_02404cbf892f41ac8aadc18fd57e1364,766196,76696,,
17,test_0269da701f019bbca1db3ba93e8933a5,902681,90268,,
18,test_027635ab76577ce37689bd6e91c323f2,626611,62661,,
19,test_02b868c9d4630ad2167f70693a579868,587472,58472,,


target — ошибки по изображениям:


,image_id,label,prediction,error,error_detail
0,33ef91d464c03eb9,894412,8D4z2,,
1,41ada5001f1e1e88,432077,4267,,
2,5cb6a434364bfe27,558796,55878,,
3,7619f1f1b372df6d,181263,112e3,,
4,8731a1e555abd295,926325,g263,,
5,d4c0652824fb2dfd,639125,6bi2s,,
6,d4d1163a8ce99293,642044,6e,,


,dataset,n,correct,exact_match,ci95_low,ci95_high,cer,errors,missing,median_ms,p95_ms,success_median_ms,success_p95_ms
0,holdout,1984,881,0.444052,0.422318,0.466003,0.170279,0,0,30.4273,41.29416,30.4273,41.29416
1,target,7,0,0.000000,0.000000,0.354330,0.523810,0,0,33.3312,40.01595,33.3312,40.01595


## Вывод по гипотезе

Exact match и CER вычисляются по скачанным меткам. Вердикт получается из заранее заданного критерия и интервала разности; отсутствие статистической определённости явно показывается. Правильность на публичном наборе не переносится автоматически на Instagram.

In [5]:
summary=all_summaries()
display(summary[summary.model.eq('conv-transformer-base')])
hypotheses,paired=hypothesis_results()
display(hypotheses[hypotheses.hypothesis.eq('H5')])
display(paired[paired.hypothesis.eq('H5')] if not paired.empty else paired)

,model,dataset,n,correct,exact_match,ci95_low,ci95_high,cer,errors,missing,median_ms,p95_ms,success_median_ms,success_p95_ms
12,conv-transformer-base,holdout,1984,881,0.444052,0.422318,0.466003,0.170279,0,0,30.42730,41.294160,30.42730,41.294160
13,conv-transformer-base,shared_200,200,96,0.480000,0.411792,0.548962,0.155000,0,0,30.76705,46.113865,30.76705,46.113865
14,conv-transformer-base,target,7,0,0.000000,0.000000,0.354330,0.523810,0,0,33.33120,40.015950,33.33120,40.015950


,hypothesis,criterion,verdict,estimate,ci_low,ci_high
4,H5,delta conv-transformer-base-crnn-base>=2.00%,supported,0.108871,0.074188,0.143004


,hypothesis,model_a,model_b,n,wins,losses,delta,delta_ci95_low,delta_ci95_high,mcnemar_exact_p,holm_p
3,H5,conv-transformer-base,crnn-base,1984,393,177,0.108871,0.074188,0.143004,7.748850e-20,7.748850e-20
